# Landslide Risk Model: Rain and Terrain
Ranks landslide risk from the previous 14 days of rain and the terrain around a place. Main result: ROC-AUC 0.77 with rain and terrain together, against 0.72 for rain alone (section 10).

## 1. Download and inspect the NASA landslide catalogue

In [2]:
import pandas as pd                    # tables of data
import numpy as np                     # maths on numbers
import requests                        # lets Python download from websites

URLS = [                               # two addresses for the same file; we try the first, then the second
    "https://data.nasa.gov/api/views/dd9e-wu2v/rows.csv?accessType=DOWNLOAD",
    "https://data.nasa.gov/docs/legacy/Global_Landslide_Catalog_Export/Global_Landslide_Catalog_Export_rows.csv",
]

for url in URLS:                                                  # try each address one by one
    try:
        r = requests.get(url, timeout=60)                         # download the file
        r.raise_for_status()                                      # stop with an error if the download failed
        with open("landslides_raw.csv", "wb") as f:               # open a new file on Colab, ready for writing bytes
            f.write(r.content)                                    # save what we downloaded into that file
        print("Downloaded from:", url)                            # tell us which address worked
        break                                                     # stop trying, we have the file
    except Exception as e:                                        # if this address failed
        print("Failed:", url, "->", e)                            # show why, then try the next one

df = pd.read_csv("landslides_raw.csv")                            # read the file into a table

print("Table size (rows, columns):", df.shape)                    # how many landslides and how many columns
print("\nColumn names:")                                          # heading
print(list(df.columns))                                           # list every column name
print("\nFirst 3 rows:")                                          # heading
print(df.head(3).T)                                               # first 3 rows, turned sideways so it is easy to read

for col in ["landslide_trigger", "location_accuracy", "landslide_category", "landslide_size", "country_name"]:  # columns we care about
    if col in df.columns:                                         # only if the column exists
        print("\n" + col, "- most common values:")               # heading
        print(df[col].value_counts(dropna=False).head(10))        # the 10 most common values, including empty ones

if "event_date" in df.columns:                                    # check the date column
    dates = pd.to_datetime(df["event_date"], errors="coerce")     # turn text into real dates; broken ones become empty
    print("\nDate range:", dates.min(), "to", dates.max())        # earliest and latest landslide
    print("Events per year:")                                     # heading
    print(dates.dt.year.value_counts().sort_index())              # how many landslides in each year

Failed: https://data.nasa.gov/api/views/dd9e-wu2v/rows.csv?accessType=DOWNLOAD -> 404 Client Error: Not Found for url: https://data.nasa.gov/api/views/dd9e-wu2v/rows.csv?accessType=DOWNLOAD
Downloaded from: https://data.nasa.gov/docs/legacy/Global_Landslide_Catalog_Export/Global_Landslide_Catalog_Export_rows.csv
Table size (rows, columns): (11033, 31)

Column names:
['source_name', 'source_link', 'event_id', 'event_date', 'event_time', 'event_title', 'event_description', 'location_description', 'location_accuracy', 'landslide_category', 'landslide_trigger', 'landslide_size', 'landslide_setting', 'fatality_count', 'injury_count', 'storm_name', 'photo_link', 'notes', 'event_import_source', 'event_import_id', 'country_name', 'country_code', 'admin_division_name', 'admin_division_population', 'gazeteer_closest_point', 'gazeteer_distance', 'submitted_date', 'created_date', 'last_edited_date', 'longitude', 'latitude']

First 3 rows:
                                                           

## 2. Clean the catalogue

In [3]:
df = pd.read_csv("landslides_raw.csv")                             # read the raw file again from Colab's storage

df["date"] = pd.to_datetime(df["event_date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")  # text to real dates, using the exact format we saw (month/day/year hour:minute AM/PM)
print("Dates that could not be read:", df["date"].isna().sum())    # how many dates were broken

funnel = []                                                        # a list where we record how many rows are left after each filter
def log(name, table):                                              # small helper: save a name and the current row count
    funnel.append((name, len(table)))

d = df.copy()                                                      # work on a copy so the raw table stays safe
log("start", d)

d = d.dropna(subset=["latitude", "longitude", "date"])             # remove rows missing a position or a date
log("has position and date", d)

d = d[~((d["latitude"] == 0) & (d["longitude"] == 0))]             # remove rows placed at 0,0 (a typical "unknown" placeholder, in the sea)
d = d[d["latitude"].between(-90, 90) & d["longitude"].between(-180, 180)]  # remove impossible coordinates
log("valid coordinates", d)

d = d[d["date"] >= "2007-01-01"]                                   # keep 2007 onwards, because earlier years are almost empty
log("2007 or later", d)

rain_triggers = ["downpour", "rain", "continuous_rain", "tropical_cyclone", "monsoon"]  # triggers where rainfall is the cause
d = d[d["landslide_trigger"].isin(rain_triggers)]                  # keep only those
log("rain-related trigger", d)

d = d[d["location_accuracy"].isin(["exact", "1km", "5km"])]        # keep only events placed within 5 km of the true spot
log("location within 5 km", d)

d["day"] = d["date"].dt.normalize()                                # keep the date only, drop the time of day
d["lat_r"] = d["latitude"].round(2)                                # round latitude to 2 decimals (about 1 km)
d["lon_r"] = d["longitude"].round(2)                               # round longitude the same way
d = d.drop_duplicates(subset=["lat_r", "lon_r", "day"])            # keep one event per place per day, so a single storm does not count 50 times
log("duplicates removed", d)

clean = d[["event_id", "day", "latitude", "longitude", "landslide_trigger",
           "location_accuracy", "landslide_category", "landslide_size",
           "fatality_count", "country_name"]].copy()               # keep only the columns we will use
clean["landslide"] = 1                                             # label: 1 means "a landslide happened here on this day"
clean = clean.reset_index(drop=True)                               # renumber rows from 0

print("\nRows left after each filter:")                            # heading
for name, n in funnel:                                             # print the funnel
    print(f"  {name:28s} {n}")

print("\nFinal table size:", clean.shape)                          # rows and columns of the clean table
print("\nEvents per year:")                                        # heading
print(clean["day"].dt.year.value_counts().sort_index())            # events per year
print("\nTop 10 countries:")                                       # heading
print(clean["country_name"].value_counts(dropna=False).head(10))   # top countries
print("\nLocation accuracy:")                                      # heading
print(clean["location_accuracy"].value_counts())                   # how exact the places are
print("\nTrigger:")                                                # heading
print(clean["landslide_trigger"].value_counts())                   # which triggers remain

clean.to_csv("landslides_clean.csv", index=False)                  # save our clean table

Dates that could not be read: 0

Rows left after each filter:
  start                        11033
  has position and date        11033
  valid coordinates            11033
  2007 or later                10988
  rain-related trigger         8672
  location within 5 km         4904
  duplicates removed           4809

Final table size: (4809, 11)

Events per year:
day
2007    107
2008    113
2009    239
2010    795
2011    632
2012    331
2013    493
2014    392
2015    493
2016    463
2017    751
Name: count, dtype: int64

Top 10 countries:
country_name
United States     1374
NaN                904
India              536
Philippines        233
Nepal              159
Indonesia          148
Brazil             129
United Kingdom     125
Malaysia            89
China               85
Name: count, dtype: int64

Location accuracy:
location_accuracy
5km      2678
1km      1704
exact     427
Name: count, dtype: int64

Trigger:
landslide_trigger
downpour            2578
rain                1505


## 3. Create "no landslide" examples

In [4]:
clean = pd.read_csv("landslides_clean.csv", parse_dates=["day"])   # load the clean landslides from Step 2
rng = np.random.default_rng(42)                                    # random number generator; 42 makes the results repeat every time

DATE_MIN = np.datetime64("2007-01-01")                             # earliest allowed date for a fake example
DATE_MAX = np.datetime64("2017-09-28")                             # latest allowed date (the catalogue ends here)

def haversine_km(lat1, lon1, lat2, lon2):                          # distance in km between points on the Earth
    R = 6371.0                                                     # radius of the Earth in km
    p1, p2 = np.radians(lat1), np.radians(lat2)                    # latitudes in radians
    dphi = p2 - p1                                                 # latitude difference
    dlmb = np.radians(lon2 - lon1)                                 # longitude difference
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2   # haversine formula, part 1
    return 2 * R * np.arcsin(np.sqrt(a))                           # haversine formula, part 2: the distance

def move_point(lat, lon, km, bearing_deg):                         # find the point that is km away in a given direction
    R = 6371.0
    br = np.radians(bearing_deg)                                   # direction in radians (0 = north, 90 = east)
    d = km / R                                                     # distance as an angle on the globe
    la1, lo1 = np.radians(lat), np.radians(lon)                    # start point in radians
    la2 = np.arcsin(np.sin(la1) * np.cos(d) + np.cos(la1) * np.sin(d) * np.cos(br))   # new latitude
    lo2 = lo1 + np.arctan2(np.sin(br) * np.sin(d) * np.cos(la1),
                           np.cos(d) - np.sin(la1) * np.sin(la2))                     # new longitude
    return float(np.degrees(la2)), float((np.degrees(lo2) + 540) % 360 - 180)         # back to degrees; keeps longitude between -180 and 180

ev_lat = clean["latitude"].to_numpy()                              # all landslide latitudes as an array
ev_lon = clean["longitude"].to_numpy()                             # all landslide longitudes
ev_day = clean["day"].to_numpy().astype("datetime64[D]")           # all landslide dates, as whole days

def too_close_to_event(lat, lon, day):                             # True if a real landslide was recorded near this place and time
    dist = haversine_km(lat, lon, ev_lat, ev_lon)                  # distance to every real landslide
    gap = np.abs((ev_day - day).astype(int))                       # day difference to every real landslide
    return bool(np.any((dist < 15) & (gap <= 30)))                 # within 15 km and within 30 days of any of them

rows = []                                                          # list that will collect every example (real and fake)
no_temporal, no_spatial = 0, 0                                     # counters for cases where we could not make a fake example

for pid, lat, lon, day, country in zip(clean["event_id"], ev_lat, ev_lon, ev_day, clean["country_name"]):   # go through each real landslide
    rows.append({"pair_id": pid, "day": day, "latitude": lat, "longitude": lon,
                 "landslide": 1, "kind": "landslide", "country_name": country})   # the real landslide, label 1

    made = False                                                   # have we made the same-place fake yet?
    for _ in range(50):                                            # try up to 50 random dates
        shift = int(rng.integers(31, 121)) * int(rng.choice([-1, 1]))   # 31 to 120 days before or after
        nd = day + np.timedelta64(shift, "D")                      # the new date
        if DATE_MIN <= nd <= DATE_MAX and not too_close_to_event(lat, lon, nd):   # inside the date range and no recorded landslide nearby
            rows.append({"pair_id": pid, "day": nd, "latitude": lat, "longitude": lon,
                         "landslide": 0, "kind": "same_place_other_day", "country_name": country})   # fake example, label 0
            made = True
            break                                                  # done, stop trying
    if not made:
        no_temporal += 1                                           # count the failures

    made = False                                                   # have we made the other-place fake yet?
    for _ in range(50):                                            # try up to 50 random places
        km = rng.uniform(20, 150)                                  # random distance, 20 to 150 km
        bearing = rng.uniform(0, 360)                              # random direction
        nlat, nlon = move_point(lat, lon, km, bearing)             # the new place
        if abs(nlat) <= 90 and not too_close_to_event(nlat, nlon, day):   # valid latitude and no recorded landslide nearby that day
            rows.append({"pair_id": pid, "day": day, "latitude": nlat, "longitude": nlon,
                         "landslide": 0, "kind": "other_place_same_day", "country_name": country})    # fake example, label 0
            made = True
            break
    if not made:
        no_spatial += 1

points = pd.DataFrame(rows)                                        # turn the list into a table
points["day"] = pd.to_datetime(points["day"])                      # make sure the dates are real dates

print("Could not make same-place fakes:", no_temporal)             # failures
print("Could not make other-place fakes:", no_spatial)             # failures
print("\nTable size:", points.shape)                               # rows and columns
print("\nRows by kind:")                                           # heading
print(points["kind"].value_counts())                               # how many of each kind
print("\nShare of landslide rows:", round(points["landslide"].mean(), 3))   # fraction with label 1
print("\nSample rows:")                                            # heading
print(points.head(6))                                              # first 6 rows

points.to_csv("points.csv", index=False)                           # save progress

Could not make same-place fakes: 0
Could not make other-place fakes: 0

Table size: (14427, 7)

Rows by kind:
kind
landslide               4809
same_place_other_day    4809
other_place_same_day    4809
Name: count, dtype: int64

Share of landslide rows: 0.333

Sample rows:
   pair_id        day   latitude   longitude  landslide                  kind  \
0      956 2009-01-02  45.420000 -122.663000          1             landslide   
1      956 2009-02-10  45.420000 -122.663000          0  same_place_other_day   
2      956 2009-01-02  45.854404 -123.435182          0  other_place_same_day   
3     2603 2010-10-16  10.333600  123.897800          1             landslide   
4     2603 2011-01-21  10.333600  123.897800          0  same_place_other_day   
5     2603 2010-10-16  11.094242  124.704929          0  other_place_same_day   

    country_name  
0  United States  
1  United States  
2  United States  
3    Philippines  
4    Philippines  
5    Philippines  


## 4. Download rainfall from Open-Meteo (first 2,500 groups)

In [5]:
import os, time                                              # os: files and folders; time: lets us pause between calls
from google.colab import drive                               # Colab tool that connects your Google Drive

drive.mount("/content/drive")                                # connects Drive; a popup asks for permission, click Allow
SAVE_DIR = "/content/drive/MyDrive/landslide_project"        # a folder in your Drive where results are saved safely
os.makedirs(SAVE_DIR, exist_ok=True)                         # create the folder if it does not exist yet
OUT = SAVE_DIR + "/rain_features.csv"                        # the file that will hold the rainfall results

points = pd.read_csv("points.csv", parse_dates=["day"])      # load the examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"]   # a unique name for each row, like "956|landslide"

N_PAIRS = 2500                                                # how many landslide groups to process (change to 2500 for the full run)
all_pairs = np.sort(points["pair_id"].unique())              # list of all group ids
order = np.random.default_rng(42).permutation(all_pairs)     # shuffle them in a fixed order (same shuffle every time)
chosen = order[:N_PAIRS]                                     # take the first N_PAIRS; raising N later keeps the same ones and adds more
sub = points[points["pair_id"].isin(chosen)].copy()          # the rows we will process (3 rows per group)

done_keys = set()                                            # names of rows already finished
if os.path.exists(OUT):                                      # if the results file already exists
    done_keys = set(pd.read_csv(OUT)["key"])                 # read which rows are done
todo = sub[~sub["key"].isin(done_keys)]                      # rows still to do
print("Rows selected:", len(sub), "| already done:", len(done_keys & set(sub["key"])), "| to do now:", len(todo))

URL = "https://archive-api.open-meteo.com/v1/archive"        # Open-Meteo history service
buffer = []                                                  # temporary list of results, saved in batches

def flush():                                                 # saves the temporary list to the file
    if buffer:                                               # only if there is something to save
        pd.DataFrame(buffer).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)   # add rows to the end; write column names only for a new file
        buffer.clear()                                       # empty the list

for i, r in enumerate(todo.itertuples(), start=1):           # go through each row we still need
    end = r.day                                              # last day of the window = the event date
    start = end - pd.Timedelta(days=13)                      # first day = 13 days earlier, so 14 days in total
    params = {
        "latitude": r.latitude, "longitude": r.longitude,    # the place
        "start_date": start.strftime("%Y-%m-%d"),            # window start
        "end_date": end.strftime("%Y-%m-%d"),                # window end
        "daily": "precipitation_sum",                        # total rain per day, in mm
        "timezone": "auto",                                  # days follow the local time of the place
    }

    data, limit_hit = None, False                            # data = the reply; limit_hit = did we hit the free limit
    for attempt in range(3):                                 # try up to 3 times
        try:
            resp = requests.get(URL, params=params, timeout=30)   # send the request
        except requests.RequestException:                    # network problem
            time.sleep(5)                                    # wait 5 seconds
            continue                                         # try again
        if resp.status_code == 200:                          # success
            data = resp.json()                               # the reply as a dictionary
            break                                            # stop retrying
        if resp.status_code == 429:                          # 429 means "too many requests"
            limit_hit = True
            time.sleep(65)                                   # wait over a minute, then retry
            continue
        break                                                # any other error: give up on this row only

    if data is None and limit_hit:                           # still blocked after retries
        print("Free limit reached at row", i, "- progress is saved. Run this cell again later (an hour, or tomorrow).")
        break                                                # stop the whole loop safely

    row = {"key": r.key, "rain_1d": np.nan, "rain_3d": np.nan,
           "rain_7d": np.nan, "rain_14d": np.nan, "wet_days_14d": np.nan}   # start with empty values
    rain = data.get("daily", {}).get("precipitation_sum") if data else None   # the list of 14 daily rain values
    if rain is not None and len(rain) == 14 and all(v is not None for v in rain):   # use it only if all 14 days exist
        rain = np.array(rain, dtype=float)                   # list to numbers
        row.update({
            "rain_1d": rain[-1],                             # rain on the event day itself
            "rain_3d": rain[-3:].sum(),                      # total over the last 3 days
            "rain_7d": rain[-7:].sum(),                      # total over the last 7 days
            "rain_14d": rain.sum(),                          # total over all 14 days
            "wet_days_14d": int((rain >= 1).sum()),          # how many of the 14 days had 1 mm or more
        })
    buffer.append(row)                                       # add this row to the temporary list

    if len(buffer) >= 100:                                   # every 100 rows
        flush()                                              # save to Drive
    if i % 500 == 0:                                         # every 500 rows
        print("Done", i, "of", len(todo))                    # show progress
    time.sleep(0.5)                                          # short pause so we do not go over 5,000 calls an hour

flush()                                                      # save whatever is left

feat = pd.read_csv(OUT)                                      # read all saved results
print("\nRows saved in total:", len(feat))                    # how many rows have rainfall
print("Rows with missing rain:", int(feat["rain_14d"].isna().sum()))   # rows where the data was incomplete

check = sub.merge(feat, on="key")                             # attach rainfall to the example table
print("\nAverage rain by kind (mm):")                         # heading
print(check.groupby("kind")[["rain_1d", "rain_3d", "rain_7d", "rain_14d", "wet_days_14d"]].mean().round(1))   # the sanity check

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows selected: 7500 | already done: 7500 | to do now: 0

Rows saved in total: 7500
Rows with missing rain: 31

Average rain by kind (mm):
                      rain_1d  rain_3d  rain_7d  rain_14d  wet_days_14d
kind                                                                   
landslide                22.7     55.2     94.5     149.8           9.7
other_place_same_day     18.2     44.3     78.4     125.6           9.2
same_place_other_day      4.6     14.1     33.4      65.5           6.6


## 5. Terrain, first version (500 m grid, Open-Meteo elevation API)
Earlier experiment, kept to show the process. The main result is in section 10.

In [6]:
import os, time                                                   # os: files; time: pauses
from google.colab import drive                                    # Colab tool to connect Google Drive

drive.mount("/content/drive")                                     # connect Drive (already allowed before, so no popup)
SAVE_DIR = "/content/drive/MyDrive/landslide_project"             # same folder as Step 4
RAIN_FILE = SAVE_DIR + "/rain_features.csv"                       # the rainfall results from Step 4
OUT = SAVE_DIR + "/terrain_features.csv"                          # the file for terrain results

points = pd.read_csv("points.csv", parse_dates=["day"])           # examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"]   # same row names as in Step 4
rain = pd.read_csv(RAIN_FILE)                                     # rows that have rainfall
rows = points[points["key"].isin(rain["key"])].reset_index(drop=True)   # keep only those rows
rows["loc_key"] = rows["latitude"].map("{:.5f}".format) + "|" + rows["longitude"].map("{:.5f}".format)   # a name for each place, made from its coordinates

locs = rows.drop_duplicates("loc_key")[["loc_key", "latitude", "longitude"]].reset_index(drop=True)   # one line per unique place
print("Rows:", len(rows), "| unique places:", len(locs))          # how many places we must look up

SPACING_M = 500                                                   # distance between grid points, in metres
URL = "https://api.open-meteo.com/v1/elevation"                   # Open-Meteo elevation service
ROWS_PER_CALL = 11                                                # 11 places x 9 points = 99 points, just under the limit of 100

def grid_points(lat, lon):                                        # makes the 9 points around one place
    dlat = SPACING_M / 111320.0                                   # 500 m expressed in degrees of latitude
    dlon = SPACING_M / (111320.0 * np.cos(np.radians(lat)))       # 500 m in degrees of longitude (degrees get shorter away from the equator)
    lats, lons = [], []                                           # empty lists for the 9 points
    for j in (1, 0, -1):                                          # north row, middle row, south row
        for i in (-1, 0, 1):                                      # west, middle, east
            lats.append(min(90.0, max(-90.0, lat + j * dlat)))    # latitude of this point, kept between -90 and 90
            lons.append(((lon + i * dlon + 180) % 360) - 180)     # longitude of this point, kept between -180 and 180
    return lats, lons

def terrain_from_grid(z):                                         # turns 9 heights into 3 terrain numbers
    Z = np.array(z, dtype=float).reshape(3, 3)                    # the heights as a 3x3 table
    dzdx = ((Z[0,2] + 2*Z[1,2] + Z[2,2]) - (Z[0,0] + 2*Z[1,0] + Z[2,0])) / (8 * SPACING_M)   # height change per metre, west to east (Horn's formula)
    dzdy = ((Z[2,0] + 2*Z[2,1] + Z[2,2]) - (Z[0,0] + 2*Z[0,1] + Z[0,2])) / (8 * SPACING_M)   # height change per metre, north to south
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))           # combine both directions into one slope angle in degrees
    return Z[1, 1], slope, Z.max() - Z.min()                      # centre height, slope, relief

done = set(pd.read_csv(OUT)["loc_key"]) if os.path.exists(OUT) else set()   # places already finished
todo = locs[~locs["loc_key"].isin(done)].reset_index(drop=True)   # places still to do
print("Already done:", len(done), "| to do now:", len(todo))

buffer = []                                                       # temporary list of results
def flush():                                                      # saves the list to Drive
    if buffer:
        pd.DataFrame(buffer).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        buffer.clear()

for start in range(0, len(todo), ROWS_PER_CALL):                  # go through the places in groups of 11
    chunk = todo.iloc[start:start + ROWS_PER_CALL]                # this group
    all_lat, all_lon = [], []                                     # the 99 points of this group
    for r in chunk.itertuples():
        la, lo = grid_points(r.latitude, r.longitude)             # 9 points for this place
        all_lat += la                                             # add them to the group list
        all_lon += lo
    params = {"latitude": ",".join(f"{v:.5f}" for v in all_lat),  # all latitudes joined by commas
              "longitude": ",".join(f"{v:.5f}" for v in all_lon)} # all longitudes joined by commas

    elev, limit_hit = None, False                                 # elev = the reply; limit_hit = free limit reached
    for attempt in range(3):                                      # up to 3 tries
        try:
            resp = requests.get(URL, params=params, timeout=60)   # send the request
        except requests.RequestException:                         # network problem
            time.sleep(5)
            continue
        if resp.status_code == 200:                               # success
            elev = resp.json().get("elevation")                   # the list of heights
            break
        if resp.status_code == 429:                               # too many requests
            limit_hit = True
            time.sleep(65)
            continue
        break                                                     # other error: give up on this group

    if elev is None and limit_hit:                                # blocked even after waiting
        print("Free limit reached - progress is saved. Run this cell again later.")
        break
    if elev is None or len(elev) != len(chunk) * 9:               # this group failed for another reason
        continue                                                  # skip it; a rerun will try it again

    for k, r in enumerate(chunk.itertuples()):                    # split the answer back into places
        z = elev[k * 9:(k + 1) * 9]                               # the 9 heights of place number k
        row = {"loc_key": r.loc_key, "elevation_m": np.nan, "slope_deg": np.nan, "relief_m": np.nan}   # empty result
        if all(v is not None for v in z):                         # only if all 9 heights exist
            e, s, rl = terrain_from_grid(z)                       # calculate terrain numbers
            row.update({"elevation_m": e, "slope_deg": s, "relief_m": rl})
        buffer.append(row)

    if len(buffer) >= 200:                                        # every 200 places
        flush()                                                   # save
    time.sleep(0.3)                                               # short pause between calls

flush()                                                           # save what is left

terr = pd.read_csv(OUT)                                           # all saved terrain results
print("\nPlaces saved:", len(terr), "| with missing terrain:", int(terr["slope_deg"].isna().sum()))
check = rows.merge(terr, on="loc_key")                            # attach terrain to every row
print("\nAverage terrain by kind:")
print(check.groupby("kind")[["elevation_m", "slope_deg", "relief_m"]].mean().round(1))
print("\nRows at or below sea level, by kind:")
print(check[check["elevation_m"] <= 0]["kind"].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 7500 | unique places: 4995
Already done: 4995 | to do now: 0

Places saved: 4995 | with missing terrain: 0

Average terrain by kind:
                      elevation_m  slope_deg  relief_m
kind                                                  
landslide                   629.6        6.4     186.0
other_place_same_day        761.3        5.1     141.3
same_place_other_day        629.6        6.4     186.0

Rows at or below sea level, by kind:
kind
other_place_same_day    578
landslide                27
same_place_other_day     27
Name: count, dtype: int64


## 5b. Install the map-reading library

In [7]:
!pip install -q rasterio

## 5c. Terrain, finishing with the direct Copernicus tiles
The Open-Meteo elevation API reached its free limit, so the remaining places were read from the same Copernicus terrain data on free AWS open-data storage.

In [8]:
import os, time                                                      # files and pauses
import rasterio                                                      # library that reads map-image files (GeoTIFF)
from google.colab import drive                                       # connect Google Drive

drive.mount("/content/drive")                                        # connect Drive
SAVE_DIR = "/content/drive/MyDrive/landslide_project"                # same folder as before
RAIN_FILE = SAVE_DIR + "/rain_features.csv"                          # rainfall results from Step 4
OUT = SAVE_DIR + "/terrain_features.csv"                             # terrain results (we keep adding to this file)

points = pd.read_csv("points.csv", parse_dates=["day"])              # examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"] # same row names as before
rain = pd.read_csv(RAIN_FILE)                                        # rows that have rainfall
rows = points[points["key"].isin(rain["key"])].reset_index(drop=True)   # keep only those rows
rows["loc_key"] = rows["latitude"].map("{:.5f}".format) + "|" + rows["longitude"].map("{:.5f}".format)   # a name for each place
locs = rows.drop_duplicates("loc_key")[["loc_key", "latitude", "longitude"]].reset_index(drop=True)      # one line per unique place

SPACING_M = 500                                                      # distance between grid points, in metres
BASE = "https://copernicus-dem-90m.s3.amazonaws.com"                 # the public storage address of the terrain data

tile_list = requests.get(BASE + "/tileList.txt", timeout=60).text.split()          # list of every tile that exists (ocean has none)
tiles = {t.strip("/").replace(".tif", "").split("/")[-1] for t in tile_list}        # clean each line to just the tile name
print("Tiles in list:", len(tiles), "| example:", next(iter(tiles)))                # check the list looks right

def tile_name(lat, lon):                                             # works out which tile covers a point
    la, lo = int(np.floor(lat)), int(np.floor(lon))                  # round down to whole degrees
    ns = "N" if la >= 0 else "S"                                     # north or south
    ew = "E" if lo >= 0 else "W"                                     # east or west
    return f"Copernicus_DSM_COG_30_{ns}{abs(la):02d}_00_{ew}{abs(lo):03d}_00_DEM"   # the tile's name

assert tile_name(19.07, 72.87) in tiles, "Tile list format is different from expected - send me a screenshot"   # a known land point must be found

cache = {}                                                           # remembers tiles we already opened
def elevation_at(lat, lon):                                          # height in metres at one point
    name = tile_name(lat, lon)                                       # which tile
    if name not in tiles:                                            # no tile exists = ocean
        return 0.0                                                   # sea level
    if name not in cache:                                            # not opened yet
        if len(cache) > 60:                                          # keep memory small
            for d in cache.values(): d.close()                       # close the old ones
            cache.clear()
        cache[name] = rasterio.open(f"/vsicurl/{BASE}/{name}/{name}.tif")   # open the tile over the internet
    v = float(next(cache[name].sample([(lon, lat)]))[0])             # read the height at this point
    return np.nan if np.isnan(v) else v                              # empty value stays empty

def grid_points(lat, lon):                                           # the 9 points around one place (same as before)
    dlat = SPACING_M / 111320.0
    dlon = SPACING_M / (111320.0 * np.cos(np.radians(lat)))
    lats, lons = [], []
    for j in (1, 0, -1):
        for i in (-1, 0, 1):
            lats.append(min(90.0, max(-90.0, lat + j * dlat)))
            lons.append(((lon + i * dlon + 180) % 360) - 180)
    return lats, lons

def terrain_from_grid(z):                                            # 9 heights -> elevation, slope, relief (same as before)
    Z = np.array(z, dtype=float).reshape(3, 3)
    dzdx = ((Z[0,2] + 2*Z[1,2] + Z[2,2]) - (Z[0,0] + 2*Z[1,0] + Z[2,0])) / (8 * SPACING_M)
    dzdy = ((Z[2,0] + 2*Z[2,1] + Z[2,2]) - (Z[0,0] + 2*Z[0,1] + Z[0,2])) / (8 * SPACING_M)
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
    return Z[1, 1], slope, Z.max() - Z.min()

done = set(pd.read_csv(OUT)["loc_key"]) if os.path.exists(OUT) else set()     # places already finished
todo = locs[~locs["loc_key"].isin(done)].copy()                      # places still to do
todo["tile"] = [tile_name(a, b) for a, b in zip(todo["latitude"], todo["longitude"])]   # each place's tile
todo = todo.sort_values("tile").reset_index(drop=True)               # sort by tile, so places on the same tile are read together (faster)
print("Already done:", len(done), "| to do now:", len(todo))

buffer, failures = [], 0                                             # temporary results and a failure counter
def flush():                                                         # saves the list to Drive
    if buffer:
        pd.DataFrame(buffer).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        buffer.clear()

env = rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif")   # settings that cut useless web requests
with env:
    for i, r in enumerate(todo.itertuples(), start=1):               # go through each place
        la, lo = grid_points(r.latitude, r.longitude)                # its 9 points
        try:
            z = [elevation_at(a, b) for a, b in zip(la, lo)]         # read the 9 heights
        except Exception as e:                                       # a read failed
            failures += 1                                            # count it, skip it; a rerun will retry
            continue
        row = {"loc_key": r.loc_key, "elevation_m": np.nan, "slope_deg": np.nan, "relief_m": np.nan}
        if not any(np.isnan(v) for v in z):                          # only if all 9 heights exist
            e, s, rl = terrain_from_grid(z)
            row.update({"elevation_m": e, "slope_deg": s, "relief_m": rl})
        buffer.append(row)
        if len(buffer) >= 200: flush()                               # save every 200 places
        if i % 200 == 0: print("Done", i, "of", len(todo))           # progress
    flush()

print("\nFailures (will retry on next run):", failures)

terr = pd.read_csv(OUT)                                              # all saved results
print("Places saved:", len(terr), "of", len(locs), "| with missing terrain:", int(terr["slope_deg"].isna().sum()))

test = terr.head(30).merge(locs, on="loc_key")                       # 30 places that the API calculated earlier
with env:
    diffs = [abs(elevation_at(r.latitude, r.longitude) - r.elevation_m) for r in test.itertuples()]   # compare tile height with API height
print("Check vs earlier API values (30 places): median difference", round(float(np.median(diffs)), 1), "m, max", round(float(np.max(diffs)), 1), "m")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tiles in list: 26475 | example: Copernicus_DSM_COG_30_N61_00_E172_00_DEM
Already done: 4995 | to do now: 0

Failures (will retry on next run): 0
Places saved: 4995 of 4995 | with missing terrain: 0
Check vs earlier API values (30 places): median difference 1.2 m, max 55.2 m


## 6. First model test (2,500 groups, 500 m terrain)
Earlier experiment, kept to show the process. The main result is in section 10.

In [9]:
import lightgbm as lgb                                               # the LightGBM library
from google.colab import drive                                       # Drive connection
from sklearn.model_selection import GroupShuffleSplit                # splits data so that whole groups stay together
from sklearn.linear_model import LogisticRegression                  # simple baseline model
from sklearn.preprocessing import StandardScaler                     # puts clues on a similar scale
from sklearn.pipeline import make_pipeline                           # chains scaling and model
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score   # scoring tools

drive.mount("/content/drive")                                        # connect Drive
SAVE_DIR = "/content/drive/MyDrive/landslide_project"                # our project folder

points = pd.read_csv("points.csv", parse_dates=["day"])              # examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"] # row names, as in Step 4
points["loc_key"] = points["latitude"].map("{:.5f}".format) + "|" + points["longitude"].map("{:.5f}".format)   # place names, as in Step 5
rain = pd.read_csv(SAVE_DIR + "/rain_features.csv")                  # rainfall results
terr = pd.read_csv(SAVE_DIR + "/terrain_features.csv")               # terrain results

data = points.merge(rain, on="key").merge(terr, on="loc_key")        # one table with everything
data["rain_before"] = data["rain_14d"] - data["rain_3d"]             # rain in the 11 days before the last 3 days (how wet the ground already was)

RAIN_COLS = ["rain_1d", "rain_3d", "rain_7d", "rain_14d", "wet_days_14d", "rain_before"]   # rain clues
TERRAIN_COLS = ["elevation_m", "slope_deg", "relief_m"]              # terrain clues
ALL_COLS = RAIN_COLS + TERRAIN_COLS                                  # all clues together

n0 = len(data)                                                       # rows before cleaning
data = data.dropna(subset=ALL_COLS)                                  # remove rows with any missing clue
n1 = len(data)
data = data[data["elevation_m"] > 0].reset_index(drop=True)          # remove points at or below sea level (real and fake)
n2 = len(data)
print("Rows: joined", n0, "| after removing missing", n1, "| after removing sea level", n2)

grp = data.groupby("pair_id")[["latitude", "longitude"]].first()     # one position per landslide group
grp["block"] = (np.floor(grp["latitude"] / 5).astype(int).astype(str) + "_" +
                np.floor(grp["longitude"] / 5).astype(int).astype(str))   # name of the 5-degree block the group sits in
data["block"] = data["pair_id"].map(grp["block"])                    # give every row its group's block

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42) # picks about 15% of the blocks for testing
tv_idx, te_idx = next(gss.split(data, groups=data["block"]))         # row positions for (train+validation) and test
trainval, test = data.iloc[tv_idx], data.iloc[te_idx]                # the two tables
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)   # 17.65% of the remaining 85% is about 15% of everything
tr_idx, va_idx = next(gss2.split(trainval, groups=trainval["block"]))     # split the rest into train and validation
train, val = trainval.iloc[tr_idx], trainval.iloc[va_idx]

print("\nSplit by region:")
for name, part in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"  {name:11s} rows {len(part):5d} | blocks {part['block'].nunique():3d} | landslide share {part['landslide'].mean():.3f}")

def score(name, y, p, with_f1=True):                                 # prints the scores of one model
    line = f"  {name:34s} ROC-AUC {roc_auc_score(y, p):.3f} | PR-AUC {average_precision_score(y, p):.3f}"
    if with_f1:
        line += f" | F1@0.5 {f1_score(y, (p >= 0.5).astype(int)):.3f}"   # F1 when we say "landslide" at 50% or more
    print(line)

def fit_lgb(cols):                                                   # trains a LightGBM model on the chosen clues
    m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=30,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)   # small, careful trees (little data)
    m.fit(train[cols], train["landslide"], eval_set=[(val[cols], val["landslide"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])         # stop when validation stops improving
    return m

models = {}                                                          # name -> (model, clue list)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))   # baseline: scale, then logistic regression
lr.fit(train[ALL_COLS], train["landslide"])
models["Logistic regression (all clues)"] = (lr, ALL_COLS)
models["LightGBM rain only"] = (fit_lgb(RAIN_COLS), RAIN_COLS)
models["LightGBM terrain only"] = (fit_lgb(TERRAIN_COLS), TERRAIN_COLS)
models["LightGBM all clues"] = (fit_lgb(ALL_COLS), ALL_COLS)

for title, part in [("VALIDATION", val), ("TEST (never used for any choice)", test)]:
    print(f"\n=== {title} ===")
    score("3-day rain total, no model", part["landslide"], part["rain_3d"], with_f1=False)   # one raw number as a score
    for name, (m, cols) in models.items():
        score(name, part["landslide"], m.predict_proba(part[cols])[:, 1])

print("\n=== TEST: landslide against each kind of fake (ROC-AUC) ===")
print(f"  {'model':34s} {'same place, other day':>22s} {'other place, same day':>22s}")
for name, (m, cols) in models.items():
    p = m.predict_proba(test[cols])[:, 1]                            # model scores on the test table
    res = []
    for k in ["same_place_other_day", "other_place_same_day"]:
        mask = test["kind"].isin(["landslide", k]).to_numpy()        # keep landslides plus one kind of fake
        res.append(roc_auc_score(test["landslide"].to_numpy()[mask], p[mask]))
    print(f"  {name:34s} {res[0]:22.3f} {res[1]:22.3f}")

final, _ = models["LightGBM all clues"]                              # the main model
p_test = final.predict_proba(test[ALL_COLS])[:, 1]
rng = np.random.default_rng(0)                                       # random generator for the uncertainty check
y_t = test["landslide"].to_numpy()
aucs = []
for _ in range(300):                                                 # repeat 300 times
    idx = rng.integers(0, len(y_t), len(y_t))                        # resample the test rows with replacement
    if len(np.unique(y_t[idx])) == 2:
        aucs.append(roc_auc_score(y_t[idx], p_test[idx]))
print("\nTest ROC-AUC, 95% range from resampling:", np.round(np.percentile(aucs, [2.5, 97.5]), 3))

imp = pd.Series(final.booster_.feature_importance(importance_type="gain"), index=ALL_COLS)   # how much each clue helped
print("\nFeature importance (% of total):")
print((imp / imp.sum() * 100).sort_values(ascending=False).round(1))

data.to_csv(SAVE_DIR + "/dataset_clean.csv", index=False)            # save the clean table for the next steps
final.booster_.save_model(SAVE_DIR + "/landslide_model.txt")         # save the model for the map app

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: joined 7500 | after removing missing 7469 | after removing sea level 6841

Split by region:
  Train       rows  4472 | blocks 161 | landslide share 0.364
  Validation  rows  1037 | blocks  35 | landslide share 0.354
  Test        rows  1332 | blocks  35 | landslide share 0.349

=== VALIDATION ===
  3-day rain total, no model         ROC-AUC 0.740 | PR-AUC 0.586
  Logistic regression (all clues)    ROC-AUC 0.721 | PR-AUC 0.587 | F1@0.5 0.440
  LightGBM rain only                 ROC-AUC 0.739 | PR-AUC 0.571 | F1@0.5 0.000
  LightGBM terrain only              ROC-AUC 0.593 | PR-AUC 0.427 | F1@0.5 0.000
  LightGBM all clues                 ROC-AUC 0.759 | PR-AUC 0.623 | F1@0.5 0.509

=== TEST (never used for any choice) ===
  3-day rain total, no model         ROC-AUC 0.699 | PR-AUC 0.527
  Logistic regression (all clues)    ROC-AUC 0.691 | PR-AUC 0.530 | F

## 7. Richer terrain clues (2 km area around each place)
Earlier experiment, kept to show the process. The main result is in section 10.

In [10]:
import os, warnings                                                   # files; warnings control
import rasterio                                                       # reads map-image files
from rasterio.windows import Window                                   # lets us read a rectangle of the image
from google.colab import drive                                        # Drive connection
warnings.filterwarnings("ignore")                                     # hides harmless maths warnings about empty areas

drive.mount("/content/drive")                                         # connect Drive
SAVE_DIR = "/content/drive/MyDrive/landslide_project"                 # our project folder
OUT = SAVE_DIR + "/terrain_v2.csv"                                    # new terrain results

points = pd.read_csv("points.csv", parse_dates=["day"])               # all examples from Step 3
points["loc_key"] = points["latitude"].map("{:.5f}".format) + "|" + points["longitude"].map("{:.5f}".format)   # place names, as before
locs = points.drop_duplicates("loc_key")[["loc_key", "latitude", "longitude"]].reset_index(drop=True)          # one line per unique place
print("Unique places:", len(locs))

BASE = "https://copernicus-dem-90m.s3.amazonaws.com"                  # public terrain storage
tile_list = requests.get(BASE + "/tileList.txt", timeout=60).text.split()          # tiles that exist (ocean has none)
tiles = {t.strip("/").replace(".tif", "").split("/")[-1] for t in tile_list}        # just the tile names

def tile_name(lat, lon):                                              # which 1-degree tile covers a point
    la, lo = int(np.floor(lat)), int(np.floor(lon))
    ns, ew = ("N" if la >= 0 else "S"), ("E" if lo >= 0 else "W")
    return f"Copernicus_DSM_COG_30_{ns}{abs(la):02d}_00_{ew}{abs(lo):03d}_00_DEM"

R_NEAR, R_FAR = 500, 2000                                             # the two neighbourhood sizes, in metres
COLS = ["elevation_m", "slope_500m", "slope_mean_2km", "slope_p90_2km", "steep_frac_2km", "relief_2km", "tpi_2km"]   # our 7 clues

cache = {}                                                            # tiles we have already opened
def get_tile(name):                                                   # opens a tile, or reuses it
    if name not in cache:
        if len(cache) > 40:                                           # keep memory small
            for d in cache.values(): d.close()
            cache.clear()
        cache[name] = rasterio.open(f"/vsicurl/{BASE}/{name}/{name}.tif")
    return cache[name]

def terrain_features(lat, lon):                                       # all 7 clues for one place
    empty = {c: np.nan for c in COLS}                                 # a result full of empty values
    name = tile_name(lat, lon)
    if name not in tiles:                                             # no tile = ocean
        empty["elevation_m"] = 0.0                                    # sea level
        return empty
    ds = get_tile(name)
    xres, yres = ds.res                                               # size of one pixel, in degrees
    dy = yres * 111320.0                                              # pixel height in metres
    dx = xres * 111320.0 * np.cos(np.radians(lat))                    # pixel width in metres (shrinks away from the equator)
    ny, nx = int(np.ceil(R_FAR / dy)), int(np.ceil(R_FAR / dx))       # how many pixels reach 2 km up/down and left/right
    row, col = ds.index(lon, lat)                                     # the pixel that holds our point
    win = Window(col - nx, row - ny, 2 * nx + 1, 2 * ny + 1)          # the rectangle around it
    Z = ds.read(1, window=win, boundless=True, fill_value=np.nan, out_dtype="float32")   # heights in that rectangle; outside the tile = empty

    yy = (np.arange(Z.shape[0]) - ny) * dy                            # distance north-south of each row from the centre, in metres
    xx = (np.arange(Z.shape[1]) - nx) * dx                            # distance east-west of each column from the centre
    D = np.hypot(yy[:, None], xx[None, :])                            # distance of every pixel from the centre
    gy, gx = np.gradient(Z, dy, dx)                                   # height change per metre in both directions
    S = np.degrees(np.arctan(np.hypot(gy, gx)))                       # slope angle of every pixel, in degrees
    near, far = D <= R_NEAR, D <= R_FAR                               # yes/no masks for the two circles

    if np.isnan(Z[ny, nx]) or np.isfinite(Z[far]).mean() < 0.5 or not np.isfinite(S[near]).any():   # not enough valid pixels
        return empty
    sv = S[far][np.isfinite(S[far])]                                  # valid slopes inside the 2 km circle
    return {
        "elevation_m": float(Z[ny, nx]),                              # height at the point
        "slope_500m": float(np.nanmean(S[near])),                     # mean slope within 500 m
        "slope_mean_2km": float(np.nanmean(S[far])),                  # mean slope within 2 km
        "slope_p90_2km": float(np.percentile(sv, 90)),                # steep end of the slopes: 90th percentile
        "steep_frac_2km": float((sv > 15).mean()),                    # share of pixels steeper than 15 degrees
        "relief_2km": float(np.nanmax(Z[far]) - np.nanmin(Z[far])),   # highest minus lowest
        "tpi_2km": float(Z[ny, nx] - np.nanmean(Z[far])),             # point height minus neighbourhood average
    }

done = set(pd.read_csv(OUT)["loc_key"]) if os.path.exists(OUT) else set()   # places already finished
todo = locs[~locs["loc_key"].isin(done)].copy()                       # places still to do
todo["tile"] = [tile_name(a, b) for a, b in zip(todo["latitude"], todo["longitude"])]
todo = todo.sort_values("tile").reset_index(drop=True)                # same-tile places together = faster
print("Already done:", len(done), "| to do now:", len(todo))

buffer, failures = [], 0
def flush():                                                          # saves results to Drive
    if buffer:
        pd.DataFrame(buffer).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        buffer.clear()

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"):   # fewer useless web requests
    for i, r in enumerate(todo.itertuples(), start=1):
        try:
            feats = terrain_features(r.latitude, r.longitude)         # the 7 clues
        except Exception:                                             # a read failed
            failures += 1                                             # skip it; a rerun retries it
            continue
        feats["loc_key"] = r.loc_key
        buffer.append(feats)
        if len(buffer) >= 200: flush()                                # save every 200 places
        if i % 200 == 0: print("Done", i, "of", len(todo))
    flush()

print("\nFailures (retry by running again):", failures)
terr = pd.read_csv(OUT)
print("Places saved:", len(terr), "of", len(locs))
print("Missing values per clue:")
print(terr[COLS].isna().sum())

old = pd.read_csv(SAVE_DIR + "/terrain_features.csv")                 # the earlier terrain results
both = terr.merge(old, on="loc_key")                                  # places in both files
both = both[both["elevation_m_x"] > 0]                                # land only
print("\nAgreement with the earlier terrain (", len(both), "places ):")
print("  median elevation difference:", round(float((both["elevation_m_x"] - both["elevation_m_y"]).abs().median()), 1), "m")
print("  correlation of slope_500m with the earlier slope:", round(float(both["slope_500m"].corr(both["slope_deg"])), 2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unique places: 9605
Already done: 0 | to do now: 9605
Done 200 of 9605
Done 400 of 9605
Done 600 of 9605
Done 800 of 9605
Done 1000 of 9605
Done 1200 of 9605
Done 1400 of 9605
Done 1600 of 9605
Done 1800 of 9605
Done 2000 of 9605
Done 2200 of 9605
Done 2400 of 9605
Done 2600 of 9605
Done 2800 of 9605
Done 3000 of 9605
Done 3200 of 9605
Done 3400 of 9605
Done 3600 of 9605
Done 3800 of 9605
Done 4000 of 9605
Done 4200 of 9605
Done 4400 of 9605
Done 4600 of 9605
Done 4800 of 9605
Done 5000 of 9605
Done 5200 of 9605
Done 5400 of 9605
Done 5600 of 9605
Done 5800 of 9605
Done 6000 of 9605
Done 6200 of 9605
Done 6400 of 9605
Done 6600 of 9605
Done 6800 of 9605
Done 7000 of 9605
Done 7200 of 9605
Done 7400 of 9605
Done 7600 of 9605
Done 7800 of 9605
Done 8000 of 9605
Done 8200 of 9605
Done 8400 of 9605
Done 8600 of 9605
Done 8800 of 9605
Done 9000 of 9605
Done 9200 o

## 8. Second model test (richer terrain)
Earlier experiment, kept to show the process. The main result is in section 10.

In [11]:
import lightgbm as lgb                                               # the LightGBM library
from google.colab import drive                                       # Drive connection
from sklearn.model_selection import GroupShuffleSplit                # splits data so that whole groups stay together
from sklearn.linear_model import LogisticRegression                  # simple baseline model
from sklearn.preprocessing import StandardScaler                     # puts clues on a similar scale
from sklearn.pipeline import make_pipeline                           # chains scaling and model
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score   # scoring tools

drive.mount("/content/drive")                                        # connect Drive
SAVE_DIR = "/content/drive/MyDrive/landslide_project"                # our project folder

points = pd.read_csv("points.csv", parse_dates=["day"])              # examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"] # row names, as in Step 4
points["loc_key"] = points["latitude"].map("{:.5f}".format) + "|" + points["longitude"].map("{:.5f}".format)   # place names, as in Step 5
rain = pd.read_csv(SAVE_DIR + "/rain_features.csv")                  # rainfall results
terr = pd.read_csv(SAVE_DIR + "/terrain_v2.csv")                     # CHANGE 1: the new terrain results

data = points.merge(rain, on="key").merge(terr, on="loc_key")        # one table with everything
data["rain_before"] = data["rain_14d"] - data["rain_3d"]             # rain in the 11 days before the last 3 days (how wet the ground already was)

RAIN_COLS = ["rain_1d", "rain_3d", "rain_7d", "rain_14d", "wet_days_14d", "rain_before"]   # rain clues
TERRAIN_COLS = ["elevation_m", "slope_500m", "slope_mean_2km", "slope_p90_2km",
                "steep_frac_2km", "relief_2km", "tpi_2km"]           # CHANGE 2: the 7 new terrain clues
ALL_COLS = RAIN_COLS + TERRAIN_COLS                                  # all clues together

n0 = len(data)                                                       # rows before cleaning
data = data.dropna(subset=ALL_COLS)                                  # remove rows with any missing clue
n1 = len(data)
data = data[data["elevation_m"] > 0].reset_index(drop=True)          # remove points at or below sea level (real and fake)
n2 = len(data)
print("Rows: joined", n0, "| after removing missing", n1, "| after removing sea level", n2)

grp = data.groupby("pair_id")[["latitude", "longitude"]].first()     # one position per landslide group
grp["block"] = (np.floor(grp["latitude"] / 5).astype(int).astype(str) + "_" +
                np.floor(grp["longitude"] / 5).astype(int).astype(str))   # name of the 5-degree block the group sits in
data["block"] = data["pair_id"].map(grp["block"])                    # give every row its group's block

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42) # picks about 15% of the blocks for testing
tv_idx, te_idx = next(gss.split(data, groups=data["block"]))         # row positions for (train+validation) and test
trainval, test = data.iloc[tv_idx], data.iloc[te_idx]                # the two tables
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)   # 17.65% of the remaining 85% is about 15% of everything
tr_idx, va_idx = next(gss2.split(trainval, groups=trainval["block"]))     # split the rest into train and validation
train, val = trainval.iloc[tr_idx], trainval.iloc[va_idx]

print("\nSplit by region:")
for name, part in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"  {name:11s} rows {len(part):5d} | blocks {part['block'].nunique():3d} | landslide share {part['landslide'].mean():.3f}")

def score(name, y, p, with_f1=True):                                 # prints the scores of one model
    line = f"  {name:34s} ROC-AUC {roc_auc_score(y, p):.3f} | PR-AUC {average_precision_score(y, p):.3f}"
    if with_f1:
        line += f" | F1@0.5 {f1_score(y, (p >= 0.5).astype(int)):.3f}"   # F1 when we say "landslide" at 50% or more
    print(line)

def fit_lgb(cols):                                                   # trains a LightGBM model on the chosen clues
    m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=30,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)   # small, careful trees (little data)
    m.fit(train[cols], train["landslide"], eval_set=[(val[cols], val["landslide"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])         # stop when validation stops improving
    return m

models = {}                                                          # name -> (model, clue list)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))   # baseline: scale, then logistic regression
lr.fit(train[ALL_COLS], train["landslide"])
models["Logistic regression (all clues)"] = (lr, ALL_COLS)
models["LightGBM rain only"] = (fit_lgb(RAIN_COLS), RAIN_COLS)
models["LightGBM terrain only"] = (fit_lgb(TERRAIN_COLS), TERRAIN_COLS)
models["LightGBM all clues"] = (fit_lgb(ALL_COLS), ALL_COLS)

for title, part in [("VALIDATION", val), ("TEST (never used for any choice)", test)]:
    print(f"\n=== {title} ===")
    score("3-day rain total, no model", part["landslide"], part["rain_3d"], with_f1=False)   # one raw number as a score
    for name, (m, cols) in models.items():
        score(name, part["landslide"], m.predict_proba(part[cols])[:, 1])

print("\n=== TEST: landslide against each kind of fake (ROC-AUC) ===")
print(f"  {'model':34s} {'same place, other day':>22s} {'other place, same day':>22s}")
for name, (m, cols) in models.items():
    p = m.predict_proba(test[cols])[:, 1]                            # model scores on the test table
    res = []
    for k in ["same_place_other_day", "other_place_same_day"]:
        mask = test["kind"].isin(["landslide", k]).to_numpy()        # keep landslides plus one kind of fake
        res.append(roc_auc_score(test["landslide"].to_numpy()[mask], p[mask]))
    print(f"  {name:34s} {res[0]:22.3f} {res[1]:22.3f}")

final, _ = models["LightGBM all clues"]                              # the main model
p_test = final.predict_proba(test[ALL_COLS])[:, 1]                   # its test scores
p_rain = models["LightGBM rain only"][0].predict_proba(test[RAIN_COLS])[:, 1]   # rain-only model's test scores
y_t = test["landslide"].to_numpy()                                   # the true answers
rng = np.random.default_rng(0)                                       # random generator for the uncertainty checks
aucs, gains = [], []
for _ in range(300):                                                 # repeat 300 times
    idx = rng.integers(0, len(y_t), len(y_t))                        # resample the test rows with replacement
    if len(np.unique(y_t[idx])) == 2:
        a_all = roc_auc_score(y_t[idx], p_test[idx])                 # all-clues score on this resample
        a_rain = roc_auc_score(y_t[idx], p_rain[idx])                # rain-only score on the SAME resample
        aucs.append(a_all)
        gains.append(a_all - a_rain)                                 # CHANGE 4 (added): the gain of all clues over rain only
print("\nTest ROC-AUC of all clues, 95% range from resampling:", np.round(np.percentile(aucs, [2.5, 97.5]), 3))
print("Gain of all clues over rain only (ROC-AUC), 95% range:", np.round(np.percentile(gains, [2.5, 97.5]), 3))

imp = pd.Series(final.booster_.feature_importance(importance_type="gain"), index=ALL_COLS)   # how much each clue helped
print("\nFeature importance (% of total):")
print((imp / imp.sum() * 100).sort_values(ascending=False).round(1))

data.to_csv(SAVE_DIR + "/dataset_clean_v2.csv", index=False)         # CHANGE 3: new file name, so the old one stays safe
final.booster_.save_model(SAVE_DIR + "/landslide_model_v2.txt")      # CHANGE 3: new file name for the model

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: joined 7500 | after removing missing 7369 | after removing sea level 6845

Split by region:
  Train       rows  4925 | blocks 160 | landslide share 0.361
  Validation  rows   762 | blocks  35 | landslide share 0.362
  Test        rows  1158 | blocks  35 | landslide share 0.353

=== VALIDATION ===
  3-day rain total, no model         ROC-AUC 0.724 | PR-AUC 0.559
  Logistic regression (all clues)    ROC-AUC 0.720 | PR-AUC 0.590 | F1@0.5 0.546
  LightGBM rain only                 ROC-AUC 0.721 | PR-AUC 0.537 | F1@0.5 0.556
  LightGBM terrain only              ROC-AUC 0.583 | PR-AUC 0.429 | F1@0.5 0.000
  LightGBM all clues                 ROC-AUC 0.749 | PR-AUC 0.635 | F1@0.5 0.559

=== TEST (never used for any choice) ===
  3-day rain total, no model         ROC-AUC 0.679 | PR-AUC 0.491
  Logistic regression (all clues)    ROC-AUC 0.686 | PR-AUC 0.511 | F

## 9. Rainfall for all 4,809 groups

In [12]:
import os, time                                              # os: files and folders; time: lets us pause between calls
from google.colab import drive                               # Colab tool that connects your Google Drive

drive.mount("/content/drive")                                # connect Drive
SAVE_DIR = "/content/drive/MyDrive/landslide_project"        # the same project folder
OUT = SAVE_DIR + "/rain_features.csv"                        # the file that holds rainfall results (we keep adding to it)

points = pd.read_csv("points.csv", parse_dates=["day"])      # load the examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"]   # a unique name for each row

N_PAIRS = 4809                                               # CHANGED: all landslide groups (it was 2500)
all_pairs = np.sort(points["pair_id"].unique())              # list of all group ids
order = np.random.default_rng(42).permutation(all_pairs)     # the same fixed shuffle as before, so the first 2,500 are unchanged
chosen = order[:N_PAIRS]                                     # take the first N_PAIRS groups
sub = points[points["pair_id"].isin(chosen)].copy()          # the rows we will process

done_keys = set()                                            # names of rows already finished
if os.path.exists(OUT):                                      # if the results file exists
    done_keys = set(pd.read_csv(OUT)["key"])                 # read which rows are done
todo = sub[~sub["key"].isin(done_keys)]                      # rows still to do
print("Rows selected:", len(sub), "| already done:", len(done_keys & set(sub["key"])), "| to do now:", len(todo))

URL = "https://archive-api.open-meteo.com/v1/archive"        # Open-Meteo history service
buffer = []                                                  # temporary list of results

def flush():                                                 # saves the temporary list to the file
    if buffer:
        pd.DataFrame(buffer).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        buffer.clear()

for i, r in enumerate(todo.itertuples(), start=1):           # go through each row we still need
    end = r.day                                              # last day of the window = the event date
    start = end - pd.Timedelta(days=13)                      # first day = 13 days earlier, so 14 days in total
    params = {
        "latitude": r.latitude, "longitude": r.longitude,    # the place
        "start_date": start.strftime("%Y-%m-%d"),            # window start
        "end_date": end.strftime("%Y-%m-%d"),                # window end
        "daily": "precipitation_sum",                        # total rain per day, in mm
        "timezone": "auto",                                  # days follow the local time of the place
    }

    data, limit_hit = None, False                            # data = the reply; limit_hit = did we hit the free limit
    for attempt in range(3):                                 # try up to 3 times
        try:
            resp = requests.get(URL, params=params, timeout=30)   # send the request
        except requests.RequestException:                    # network problem
            time.sleep(5)
            continue
        if resp.status_code == 200:                          # success
            data = resp.json()
            break
        if resp.status_code == 429:                          # "too many requests"
            limit_hit = True
            time.sleep(65)                                   # wait over a minute, then retry
            continue
        break                                                # any other error: give up on this row only

    if data is None and limit_hit:                           # still blocked after retries
        print("Free limit reached at row", i, "- progress is saved. Run this cell again later (an hour, or tomorrow).")
        break                                                # stop safely

    row = {"key": r.key, "rain_1d": np.nan, "rain_3d": np.nan,
           "rain_7d": np.nan, "rain_14d": np.nan, "wet_days_14d": np.nan}   # start with empty values
    rain = data.get("daily", {}).get("precipitation_sum") if data else None   # the list of 14 daily rain values
    if rain is not None and len(rain) == 14 and all(v is not None for v in rain):   # use it only if all 14 days exist
        rain = np.array(rain, dtype=float)
        row.update({
            "rain_1d": rain[-1],                             # rain on the event day itself
            "rain_3d": rain[-3:].sum(),                      # total over the last 3 days
            "rain_7d": rain[-7:].sum(),                      # total over the last 7 days
            "rain_14d": rain.sum(),                          # total over all 14 days
            "wet_days_14d": int((rain >= 1).sum()),          # days with 1 mm or more
        })
    buffer.append(row)

    if len(buffer) >= 100:                                   # every 100 rows
        flush()                                              # save to Drive
    if i % 500 == 0:                                         # every 500 rows
        print("Done", i, "of", len(todo))
    time.sleep(0.5)                                          # short pause, so we stay under 5,000 calls an hour

flush()                                                      # save whatever is left

feat = pd.read_csv(OUT)
print("\nRows saved in total:", len(feat), "| with missing rain:", int(feat["rain_14d"].isna().sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows selected: 14427 | already done: 7500 | to do now: 6927
Done 500 of 6927
Done 1000 of 6927
Done 1500 of 6927
Done 2000 of 6927
Done 2500 of 6927
Done 3000 of 6927
Done 3500 of 6927
Done 4000 of 6927
Done 4500 of 6927
Done 5000 of 6927
Done 5500 of 6927
Done 6000 of 6927
Done 6500 of 6927

Rows saved in total: 14427 | with missing rain: 31


## 10. Final model: 5 repeated region splits (main result)

In [13]:
import lightgbm as lgb                                               # the LightGBM library
from google.colab import drive                                       # Drive connection
from sklearn.model_selection import GroupShuffleSplit                # splits data so that whole groups stay together
from sklearn.linear_model import LogisticRegression                  # simple baseline model
from sklearn.preprocessing import StandardScaler                     # puts clues on a similar scale
from sklearn.pipeline import make_pipeline                           # chains scaling and model
from sklearn.metrics import roc_auc_score, average_precision_score   # scoring tools

drive.mount("/content/drive")                                        # connect Drive
SAVE_DIR = "/content/drive/MyDrive/landslide_project"                # our project folder

points = pd.read_csv("points.csv", parse_dates=["day"])              # examples from Step 3
points["key"] = points["pair_id"].astype(str) + "|" + points["kind"] # row names, as in Step 4
points["loc_key"] = points["latitude"].map("{:.5f}".format) + "|" + points["longitude"].map("{:.5f}".format)   # place names
rain = pd.read_csv(SAVE_DIR + "/rain_features.csv")                  # rainfall results (now all rows)
terr = pd.read_csv(SAVE_DIR + "/terrain_v2.csv")                     # terrain results

data = points.merge(rain, on="key").merge(terr, on="loc_key")        # one table with everything
data["rain_before"] = data["rain_14d"] - data["rain_3d"]             # rain in the 11 days before the last 3 days

RAIN_COLS = ["rain_1d", "rain_3d", "rain_7d", "rain_14d", "wet_days_14d", "rain_before"]   # rain clues
TERRAIN_COLS = ["elevation_m", "slope_500m", "slope_mean_2km", "slope_p90_2km",
                "steep_frac_2km", "relief_2km", "tpi_2km"]           # terrain clues
ALL_COLS = RAIN_COLS + TERRAIN_COLS                                  # all clues together

n0 = len(data)
data = data.dropna(subset=ALL_COLS)                                  # remove rows with any missing clue
n1 = len(data)
data = data[data["elevation_m"] > 0].reset_index(drop=True)          # remove points at or below sea level (real and fake)
print("Rows: joined", n0, "| after removing missing", n1, "| after removing sea level", len(data))

grp = data.groupby("pair_id")[["latitude", "longitude"]].first()     # one position per landslide group
grp["block"] = (np.floor(grp["latitude"] / 5).astype(int).astype(str) + "_" +
                np.floor(grp["longitude"] / 5).astype(int).astype(str))   # name of the 5-degree block of each group
data["block"] = data["pair_id"].map(grp["block"])                    # give every row its group's block

def make_split(seed):                                                # makes one train / validation / test split by region
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)   # about 15% of the blocks go to test
    tv, te = next(g1.split(data, groups=data["block"]))
    trainval, test = data.iloc[tv], data.iloc[te]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=seed) # about 15% of everything goes to validation
    tr, va = next(g2.split(trainval, groups=trainval["block"]))
    return trainval.iloc[tr], trainval.iloc[va], test

LGB_PARAMS = dict(learning_rate=0.03, num_leaves=15, min_child_samples=30,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)   # small, careful trees

def fit_lgb(train, val, cols):                                       # trains LightGBM on the chosen clues
    m = lgb.LGBMClassifier(n_estimators=800, **LGB_PARAMS)
    m.fit(train[cols], train["landslide"], eval_set=[(val[cols], val["landslide"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])         # stop when validation stops improving
    return m

def eval_scores(y, p, kinds):                                        # all scores of one model on one test set
    out = {"roc": roc_auc_score(y, p), "pr": average_precision_score(y, p)}
    for k, name in [("same_place_other_day", "roc_same_place"), ("other_place_same_day", "roc_other_place")]:
        mask = kinds.isin(["landslide", k]).to_numpy()               # landslides plus one kind of fake
        out[name] = roc_auc_score(y[mask], p[mask])
    return out

results, best_iters, imps, sizes = [], [], [], []                    # collectors for the 5 repeats
for seed in range(1, 6):                                             # five different region splits
    train, val, test = make_split(seed)
    y, kinds = test["landslide"].to_numpy(), test["kind"]
    sizes.append((seed, len(train), len(val), len(test)))
    preds = {"3-day rain total, no model": test["rain_3d"].to_numpy()}   # a raw number used as a score

    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    lr.fit(train[ALL_COLS], train["landslide"])
    preds["Logistic regression (all clues)"] = lr.predict_proba(test[ALL_COLS])[:, 1]

    preds["LightGBM rain only"] = fit_lgb(train, val, RAIN_COLS).predict_proba(test[RAIN_COLS])[:, 1]
    preds["LightGBM terrain only"] = fit_lgb(train, val, TERRAIN_COLS).predict_proba(test[TERRAIN_COLS])[:, 1]
    m_all = fit_lgb(train, val, ALL_COLS)
    preds["LightGBM all clues"] = m_all.predict_proba(test[ALL_COLS])[:, 1]

    best_iters.append(m_all.best_iteration_)                         # how many trees the all-clues model used
    imps.append(pd.Series(m_all.booster_.feature_importance(importance_type="gain"), index=ALL_COLS))

    for name, p in preds.items():
        row = eval_scores(y, p, kinds)
        row.update({"repeat": seed, "model": name})
        results.append(row)

print("\nRows in (train, validation, test) for each repeat:")
for s in sizes: print("  repeat", s[0], "->", s[1:])

res = pd.DataFrame(results)                                          # all results in one table
print("\n=== TEST SCORES, mean +/- spread over 5 region splits ===")
print(f"  {'model':34s} {'ROC-AUC':>15s} {'PR-AUC':>15s} {'same place':>15s} {'other place':>15s}")
for name in res["model"].unique():
    r = res[res["model"] == name]
    cells = [f"{r[c].mean():.3f} +/- {r[c].std():.3f}" for c in ["roc", "pr", "roc_same_place", "roc_other_place"]]
    print(f"  {name:34s} {cells[0]:>15s} {cells[1]:>15s} {cells[2]:>15s} {cells[3]:>15s}")

pv = res.pivot(index="repeat", columns="model", values="roc")        # ROC-AUC with one row per repeat
gain_rain = pv["LightGBM all clues"] - pv["LightGBM rain only"]      # gain of all clues over rain only, per repeat
gain_raw = pv["LightGBM all clues"] - pv["3-day rain total, no model"]   # gain over the raw 3-day rain number
print("\nGain of all clues over rain only, per repeat:", gain_rain.round(3).tolist(),
      "| mean", round(float(gain_rain.mean()), 3), "| repeats above 0:", int((gain_rain > 0).sum()), "of 5")
print("Gain of all clues over 3-day rain total, per repeat:", gain_raw.round(3).tolist(),
      "| mean", round(float(gain_raw.mean()), 3), "| repeats above 0:", int((gain_raw > 0).sum()), "of 5")

imp = pd.concat([s / s.sum() * 100 for s in imps], axis=1).mean(axis=1).sort_values(ascending=False)
print("\nFeature importance, average over 5 repeats (% of total):")
print(imp.round(1))

n_final = int(np.mean(best_iters))                                   # tree count for the final model
final = lgb.LGBMClassifier(n_estimators=n_final, **LGB_PARAMS)       # same settings as the tests
final.fit(data[ALL_COLS], data["landslide"])                         # train the final model on ALL the data
print("\nFinal model trained on", len(data), "rows with", n_final, "trees")

data.to_csv(SAVE_DIR + "/dataset_clean_v3.csv", index=False)         # save the clean table
final.booster_.save_model(SAVE_DIR + "/landslide_model_v3.txt")      # save the model for the map app

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: joined 14427 | after removing missing 14206 | after removing sea level 13233

Rows in (train, validation, test) for each repeat:
  repeat 1 -> (7944, 2752, 2537)
  repeat 2 -> (8225, 1384, 3624)
  repeat 3 -> (9958, 1754, 1521)
  repeat 4 -> (9044, 1899, 2290)
  repeat 5 -> (8370, 1906, 2957)

=== TEST SCORES, mean +/- spread over 5 region splits ===
  model                                      ROC-AUC          PR-AUC      same place     other place
  3-day rain total, no model         0.723 +/- 0.011 0.563 +/- 0.026 0.838 +/- 0.011 0.570 +/- 0.014
  Logistic regression (all clues)    0.723 +/- 0.010 0.580 +/- 0.021 0.782 +/- 0.010 0.645 +/- 0.021
  LightGBM rain only                 0.725 +/- 0.014 0.556 +/- 0.027 0.843 +/- 0.013 0.569 +/- 0.016
  LightGBM terrain only              0.607 +/- 0.015 0.439 +/- 0.008 0.500 +/- 0.000 0.748 +/- 0.024
  Light